In [23]:
import numpy as np
import torch
import transformers
import datasets

## **Vision Transformer**
This ipynb aims to go from the most basic ready to use implementation of transformers for Vision to a completely self made, trial-and-error, implementation of the architectures. We first start out by using the ready made implementation from huggingface.

- Vision transformers (2021) propose and encoder only architecture with patches as tokens for classification tasks. Great performance gains were obserevd when the model was pretrained on a large corpus and then finetuned on smaller datasets. ([An image is worth 16x16 words](https://arxiv.org/pdf/2010.11929)).
- For generation Diffusion Transformers (DiT) are the current state of the art. Before them, autoregressive methods of generating images were devised:
    - (2018) [Image Transformer](https://arxiv.org/abs/1802.05751): Performed local attention at a pixel level, this was before patch as a token interpretation was proposed in the 2021 ViT paper.
    - (2020) [VQGAN](https://arxiv.org/abs/2012.09841): First papers to use patches as tokens for generation, still not pure transformer architecture. 

In [24]:
# Get dataset, lets start with MNIST
dataset = datasets.load_dataset("ylecun/mnist", cache_dir="./datasets/mnist")

# Train test split is inherent in huggingface dataset for MNIST
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [25]:
# Add transform to dataset to convert images to tensors and normalize them
def transform_fn(batch):
    inputs = processor([img.convert("RGB") for img in batch["image"]], return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

dataset["train"] = dataset["train"].with_transform(transform_fn)
dataset["test"] = dataset["test"].with_transform(transform_fn)

In [26]:
# Initialize the model and tokenizer
model_name = "google/vit-base-patch16-224-in21k"
model = transformers.ViTForImageClassification.from_pretrained(model_name, num_labels=10)
model

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.attention.attention.key.bias     | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.bias      | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.bias             | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.bias          | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.weight    | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.weight | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias                | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weig

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out

In [27]:
processor = transformers.AutoImageProcessor.from_pretrained(model_name)
pipeline = transformers.ImageClassificationPipeline(model=model, feature_extractor=processor, device=0 if torch.cuda.is_available() else -1)

In [28]:
# Train on the dataset
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.train()
data_collator = transformers.DefaultDataCollator()
train_args = transformers.TrainingArguments(output_dir="./results", save_strategy="epoch", learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=32, num_train_epochs=3, weight_decay=0.01, logging_dir="./logs", logging_steps=10, remove_unused_columns=False, load_best_model_at_end=True, eval_strategy="epoch")
trainer = transformers.Trainer(model, train_dataset=dataset["train"], eval_dataset=dataset["test"], args=train_args, data_collator=data_collator, processing_class = processor)
trainer.train()


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


OutOfMemoryError: CUDA out of memory. Tried to allocate 74.00 MiB. GPU 0 has a total capacity of 5.67 GiB of which 16.62 MiB is free. Including non-PyTorch memory, this process has 5.55 GiB memory in use. Of the allocated memory 5.08 GiB is allocated by PyTorch, and 364.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)